# Inspect SAE features in your sequences

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/main/cookbook/notebooks/inspect_sae_features.ipynb)

Locate sparse autoencoder activations in your IDRs and compare selected sequences.
Outputs: reusable sparse feature dataset, feature rankings, residue traces, and a comparison heatmap.

Run cells from top to bottom. In Colab select **Runtime → Change runtime type → GPU**.
A GPU is recommended; CPU works but is slower. Runtime and peak memory depend on sequence
length, model, and hardware; timings are printed below rather than promising a fixed runtime.
First use downloads model weights. Outputs are written under `OUT_DIR`; rerunning replaces
files with the same names. Download that folder from Colab before ending the session.


In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/rotskoff-group/idiom.git@v1"])

if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])

import json
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from idiom import IDiom, IDiomSAE
from idiom.data.io import Record, read_fasta, parse_idr_header
print("Python:", sys.version.split()[0])
import idiom
print("IDiom:", idiom.__file__)


# Locate the companion helper in a clone, or download it for standalone Colab use.
helper_dir = next((p for p in (Path.cwd(), Path.cwd() / "cookbook/notebooks")
                   if (p / "workflow_utils.py").is_file()), None)
if helper_dir is None:
    from urllib.request import urlretrieve
    helper_dir = Path(".idiom_notebook_helpers")
    helper_dir.mkdir(exist_ok=True)
    urlretrieve("https://raw.githubusercontent.com/rotskoff-group/idiom/main/"
                "cookbook/notebooks/workflow_utils.py", helper_dir / "workflow_utils.py")
sys.path.insert(0, str(helper_dir.resolve()))
from workflow_utils import (AA, DEMO, load_inputs, idr_sequence, isolated, check_context,
                            summaries, write_fasta, save_run)


## Inputs and validation

Use `INPUT_MODE="idr"` for FASTA records that are **already isolated IDRs** (ordinary headers
are accepted). Use `INPUT_MODE="annotated"` for full proteins: the first header token must
end in `_IDR_x-y`, with **1-based inclusive** coordinates. This notebook does not predict IDR
boundaries. Python slices use 0-based, end-exclusive coordinates.

`INPUT_FASTA=None` uses six small illustrative sequences, not experimentally labeled examples.
Set a local path to analyze your own file (upload it using the Colab Files pane).
The audit table reports rejected records and records outside the sample limit. Empty sequences,
noncanonical residues, and invalid annotations are not silently repaired. Repeated accessions
remain distinct through `record_id`; duplicate IDR sequences are reported for your review.


In [ ]:
INPUT_FASTA = None
INPUT_MODE = "idr"
MAX_RECORDS = 16
SAE = "jxliu2/idiomsae-300M-L18-k32"
DEVICE = "auto"
BATCH_SIZE = 2
FEATURE_IDS = None # None chooses the three strongest active features; or e.g. [6151, 10538, 4178]
SEQUENCE_ROWS = [0, 1, 2] # Accepted-record indices, not accession numbers
OUT_DIR = Path("sae_inspection_outputs")


In [ ]:
started = time.perf_counter()


In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
records, audit = load_inputs(INPUT_FASTA, INPUT_MODE, MAX_RECORDS)
audit.to_csv(OUT_DIR / "input_audit.csv", index=False)
display(audit)
if not records:
    raise ValueError("No accepted records. Review input_audit.csv.")
summary = summaries(records, audit)
summary.to_csv(OUT_DIR / "sequence_summary.csv", index=False)
print(f"Accepted {len(records)} records; {summary.sequence.duplicated().sum()} duplicate IDR sequences")
display(summary[["record_id", "accession", "length", "charged_fraction", "net_charge_per_residue"]])


## Encode once, reuse the dataset

The released SAE uses IDRs without flanking context and loads its own 300M host model.
Sparse datasets avoid allocating a dense residue-by-16,384-feature array. Building still holds
selected outputs in host memory; reduce `MAX_RECORDS` for large inputs and `BATCH_SIZE` for GPU memory.
The saved `sequence_index.csv` maps dataset sequence indices back to your original accessions.


In [ ]:
from idiom.sae.features import FeatureDataset
sae = IDiomSAE.from_pretrained(SAE, device=DEVICE)
if sae.fim_mode != "unprompted" or sae.region != "idr":
    raise ValueError("This notebook requires an unprompted IDR SAE.")
check_context(records, sae.model.cfg.max_seq_len)
sae.build_feature_dataset(records, OUT_DIR / "features", batch_size=BATCH_SIZE)
dataset = FeatureDataset(OUT_DIR / "features")
sequence_index = summary[["record_id", "accession", "idr_start_1based", "idr_end_1based"]].copy()
sequence_index.insert(0, "dataset_sequence", np.arange(len(records)))
sequence_index.to_csv(OUT_DIR / "sequence_index.csv", index=False)
maximum, total, count = dataset.feature_ranking()
ranking = pd.DataFrame(dict(feature_id=np.arange(dataset.num_latents), maximum=maximum,
                            total=total, firing_residues=count)).sort_values("maximum", ascending=False)
ranking.to_csv(OUT_DIR / "feature_ranking.csv", index=False)
display(ranking.head(10))


## Locate activations in your IDRs

Choose a feature by its descriptive activation ranking; this is not enrichment or a functional
annotation. Example IDs 6151, 10538, and 4178 were previously observed with proline-rich, RS-rich,
and RGG-rich patterns in the released SAE. Those observations are hypotheses for inspection,
not guarantees for a new sequence.

The stored unprompted FIM string is `132{IDR}`. Subtracting three and adding the IDR's original
start maps dataset positions to biological protein coordinates. All traces for one feature share
a vertical scale. A zero means that feature was not positively active at that residue.


In [ ]:
feature_ids = ranking.loc[ranking.firing_residues > 0, "feature_id"].head(3).tolist() if FEATURE_IDS is None else FEATURE_IDS
if any(not 0 <= f < dataset.num_latents for f in feature_ids):
    raise ValueError("Feature ID outside SAE latent range.")
selected = [i for i in SEQUENCE_ROWS if 0 <= i < len(records)]
if not selected:
    raise ValueError("Choose at least one valid SEQUENCE_ROWS entry.")
trace_rows = []
for feature in feature_ids:
    fig, axes = plt.subplots(len(selected), 1, figsize=(10, 2.2 * len(selected)), squeeze=False,
                             sharey=True, constrained_layout=True)
    for ax, seq_id in zip(axes.flat, selected):
        positions, activations = dataset.trace(seq_id, int(feature))
        protein_positions = positions - 3 + records[seq_id].idr_start + 1
        residues = [dataset.sequence(seq_id)[int(p)] for p in positions]
        ax.plot(protein_positions, activations)
        if len(positions) <= 60:
            ax.set_xticks(protein_positions, [f"{a}\n{p}" for a, p in zip(residues, protein_positions)], fontsize=7)
        ax.set(title=f"F{feature}: {sequence_index.iloc[seq_id].accession} ({records[seq_id].accession})",
               xlabel="Protein residue (1-based)", ylabel="Activation")
        trace_rows.extend(dict(record_id=records[seq_id].accession, feature_id=int(feature),
                               protein_position=int(p), residue=a, activation=float(v))
                          for p, a, v in zip(protein_positions, residues, activations))
    fig.savefig(OUT_DIR / f"feature_{feature}_traces.png", dpi=160)
    plt.show()
pd.DataFrame(trace_rows, columns=["record_id", "feature_id", "protein_position", "residue", "activation"]).to_csv(
    OUT_DIR / "residue_traces.csv", index=False)
if not feature_ids:
    print("No positive activations in this input.")


## Compare selected features across all sequences

The firing fraction is the fraction of IDR residues with positive activation. It is descriptive,
not a probability of biological function. The same scale is used across the heatmap.


In [ ]:
if feature_ids:
    fractions = np.stack([dataset.feature_stats(int(f))[2] for f in feature_ids], axis=1)
    comparison = pd.DataFrame(fractions, columns=[f"feature_{f}" for f in feature_ids])
    comparison.insert(0, "record_id", summary.record_id.to_numpy())
    comparison.to_csv(OUT_DIR / "feature_firing_fractions.csv", index=False)
    fig, ax = plt.subplots(figsize=(6, max(3, min(len(records), 30) * 0.25)), constrained_layout=True)
    im = ax.imshow(fractions[:30], aspect="auto", vmin=0, vmax=1, cmap="Oranges")
    ax.set_xticks(range(len(feature_ids)), feature_ids)
    ax.set_yticks(range(min(30, len(records))), summary.record_id[:30])
    ax.set(xlabel="Feature ID", title="Firing fraction (first 30 records shown)")
    fig.colorbar(im, ax=ax, label="Fraction of IDR residues")
    fig.savefig(OUT_DIR / "feature_comparison.png", dpi=160)
    plt.show()
save_run(OUT_DIR, dict(device=str(sae.device), input=INPUT_FASTA, mode=INPUT_MODE, max_records=MAX_RECORDS, sae=SAE,
                       batch_size=BATCH_SIZE, features=feature_ids, sequence_rows=selected), elapsed=time.perf_counter() - started)
print(f"Elapsed including model load: {time.perf_counter() - started:.1f} s")


## Optional: explore a larger reference corpus

To browse reference sequences, download a reference FASTA, set `INPUT_FASTA` to its path,
`INPUT_MODE="annotated"`, and a deliberate `MAX_RECORDS` (for example 1,000), then rerun into
a separate `OUT_DIR`. The full validation split is not needed for the starter workflow.
To reopen existing results, use `FeatureDataset(OUT_DIR / "features")` and load
`sequence_index.csv`; model inference is unnecessary for rankings and traces.

A local repository clone also provides a browser:

```bash
streamlit run src/idiom/sae/features/feature_viewer.py -- --features /absolute/path/to/features
```


## Use the outputs

Keep `input_audit.csv` with your results: `record_id` connects exported rows to the original
accession and IDR span, even when accessions repeat. `run.json` records the settings and installed
package versions. Record exact model revisions separately when freezing a published analysis.

Next: [feature enrichment](feature_enrichment.ipynb) for a background comparison, or
[steer generation](steer_generation.ipynb) to test how changing a feature affects samples.
